In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Structure Refinement: LaM(7)O3, P02.1 Synchrotron XRD

This example refines the compositionally complex LaM(7)O3 perovskite,
with La on the A site and an equimolar mixture of Ti, Cr, Mn, Fe, Co,
Ni, and Cu on the B site. The low-temperature synchrotron X-ray powder
diffraction pattern was collected at the P02.1 beamline at PETRA III.
The workflow starts from approximate structural and profile parameters,
estimates the background from the measured pattern, and improves the
model in two fitting stages.

## 🛠️ Import Library

In [2]:
import easydiffraction as edi

## 📦 Define Project

The project manages the structures, experiments, analysis, and saved
results used throughout the tutorial.

### Create Project

In [3]:
project = edi.Project(
    name='lam7o3_p021',
    description='LaM(7)O3 refinement using P02.1 synchrotron X-ray data.',
)

### Save Initial Project

Create the project directory before fitting so that analysis results
can be written as they are produced.

In [4]:
project.save_as(dir_path='projects/refine-lam7o3-p021')

Saving project 📦 'lam7o3_p021' to '../../../projects/refine-lam7o3-p021'


├── 📄 project.edi
├── 📁 structures/
├── 📁 experiments/
├── 📁 analysis/
│   └── 📄 analysis.edi
└── 📁 reports/
    └── 📄 lam7o3_p021.html


## 🧩 Define Structure

The Pnma structure is initialized from approximate values rather than
the final refined values.

### Create Structure

Define the structure as an inline CIF. Empty uncertainty parentheses,
such as `5.5()`, mark a parameter as free without assigning an initial
standard uncertainty. Values without parentheses remain fixed.

In [5]:
structure_cif = """
data_lam7o3

_cell.length_a 5.5()
_cell.length_b 7.7()
_cell.length_c 5.5()
_cell.angle_alpha 90.
_cell.angle_beta  90.
_cell.angle_gamma 90.

_space_group.name_h_m "P n m a"
_space_group.coord_system_code abc

loop_
_atom_site.id
_atom_site.type_symbol
_atom_site.fract_x
_atom_site.fract_y
_atom_site.fract_z
_atom_site.occupancy
_atom_site.adp_iso
_atom_site.adp_type
La La  0.48()  0.25   0.004()   1.        0.1() Biso
Ti Ti  0.      0.     0.        0.14286   0.1() Biso
Cr Cr  0.      0.     0.        0.14286   0.1   Biso
Mn Mn  0.      0.     0.        0.14286   0.1   Biso
Fe Fe  0.      0.     0.        0.14286   0.1   Biso
Co Co  0.      0.     0.        0.14286   0.1   Biso
Ni Ni  0.      0.     0.        0.14286   0.1   Biso
Cu Cu  0.      0.     0.        0.14286   0.1   Biso
O1 O   0.51()  0.25   0.57()    1.        0.1() Biso
O2 O   0.22()  0.03() 0.27()    1.        0.1   Biso
"""

In [6]:
project.structures.add_from_cif_str(structure_cif)

In [7]:
project.structures.show_names()

Defined structures 🧩


['lam7o3']


Use a short alias to access the structure parameters below.

In [8]:
structure = project.structures['lam7o3']

### Display Structure

Inspect the structure as text and as an interactive crystal model.

In [9]:
structure.show_as_text()

Structure 🧩 'lam7o3' as text


,Edi
1,data_lam7o3
2,
3,_cell.length_a 5.5()
4,_cell.length_b 7.7()
5,_cell.length_c 5.5()
6,_cell.angle_alpha 90.
7,_cell.angle_beta 90.
8,_cell.angle_gamma 90.
9,
10,"_space_group.name_h_m ""P n m a"""


In [10]:
project.display.structure(struct_name='lam7o3')

Structure 🧩 'lam7o3' (Atom view type: 'covalent')


## 🔬 Define Experiment

Load the measured pattern, configure the instrument and peak profile,
and link the structure to the experiment.

### Download Data

The first two columns contain 2-theta and intensity. When a third
column of standard uncertainties is absent, EasyDiffraction estimates
it from the square root of the intensity.

In [11]:
data_path = edi.download_data('meas-hep7c-xray-synchrotron', destination='data')

Getting data...


Data 'meas-hep7c-xray-synchrotron': La high-entropy perovskite (7 B-cations), synchrotron X-ray


✅ Data 'meas-hep7c-xray-synchrotron' downloaded to '../../../data/meas-hep7c-xray-synchrotron.dat'


### Create P02.1 Experiment

In [12]:
project.experiments.add_from_data_path(
    name='p021',
    data_path=data_path,
    sample_form='powder',
    beam_mode='constant wavelength',
    radiation_probe='xray',
)

⚠️ No uncertainty (sy) column provided. Defaulting to sqrt(y).


Data loaded successfully


Experiment 🔬 'p021'. Number of data points: 1459.


Use a short alias to access the P02.1 experiment parameters below.

In [13]:
experiment = project.experiments['p021']

### Set Linked Structures

Link the structural model to the measured pattern and use an
order-of-magnitude estimate for the scale factor.

In [14]:
experiment.linked_structures.create(
    structure_id='lam7o3',
    scale=0.000005,
)

### Set P02.1 Instrument Parameters

Set the monochromatic X-ray wavelength reported for the P02.1
beamline measurement and initialize the unknown 2-theta zero shift at
zero.

In [15]:
experiment.instrument.setup_wavelength = 0.207109
experiment.instrument.calib_twotheta_offset = 0.0

### Set Peak Profile

Select a pseudo-Voigt profile and provide approximate broadening
parameters. U, V, and W define the Gaussian contribution; X and Y
define the Lorentzian contribution.

In [16]:
experiment.peak.show_supported()

Peak types


,,Type,Description
1,*,pseudo-voigt,CWL pseudo-Voigt profile
2,,pseudo-voigt + berar-baldinozzi asymmetry,CWL pseudo-Voigt profile with Berar-Baldinozzi asymmetry correction.


In [17]:
experiment.peak.type = 'pseudo-voigt'

Peak profile type for experiment 'p021' changed to


pseudo-voigt


In [18]:
experiment.peak.broad_gauss_u = 0.04
experiment.peak.broad_gauss_v = -0.01
experiment.peak.broad_gauss_w = 0.001
experiment.peak.broad_lorentz_x = 0.1
experiment.peak.broad_lorentz_y = 0.0

### Set Excluded Regions

Restrict the fit to the useful measured range from 2 to 15 degrees.

In [19]:
experiment.excluded_regions.create(id='1', start=0.0, end=2.0)
experiment.excluded_regions.create(id='2', start=15.0, end=20.0)

### Set Background

Estimate initial background points from the measured pattern alone.

In [20]:
experiment.background.auto_estimate(use_model=False)

In [21]:
experiment.background.show()

Line-segment background points


,Position,Intensity
1,2.00330,584.02820
2,3.19170,616.43436
3,3.49660,608.10597
4,3.72800,594.63226
5,3.95940,574.02994
6,4.94790,457.72470
7,5.17920,443.13978
8,5.46320,433.01395
9,6.02050,432.92677
10,6.97750,421.29697


### Inspect Experiment

Display the configured experiment as text.

In [22]:
experiment.show_as_text()

Experiment 🔬 'p021' as text


,Edi
1,data_p021
2,
3,_experiment_type.sample_form powder
4,"_experiment_type.beam_mode ""constant wavelength"""
5,_experiment_type.radiation_probe xray
6,_experiment_type.scattering_type bragg
7,
8,_diffrn.ambient_temperature ?
9,_diffrn.ambient_pressure ?
10,_diffrn.ambient_magnetic_field ?


## 🚀 Perform Analysis

Select the refinement parameters, apply the shared-Biso constraints,
and improve the model in two fitting stages.

### Set Free Parameters

The independent cell lengths, selected fractional coordinates, and
three independent Biso values were marked free by `()` in the inline
CIF. They do not need to be selected again here.

Refine the scale factor, zero shift, U/V/W/X profile terms, and active
background-point intensities.

In [23]:
experiment.linked_structures['lam7o3'].scale.free = True

experiment.instrument.calib_twotheta_offset.free = True

experiment.peak.broad_gauss_u.free = True
experiment.peak.broad_gauss_v.free = True
experiment.peak.broad_gauss_w.free = True
experiment.peak.broad_lorentz_x.free = True

for point in experiment.background:
    point.intensity.free = True

Display all parameters selected for refinement.

In [24]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,lam7o3,cell,,length_a,5.50000,,-inf,inf,Å
2,lam7o3,cell,,length_b,7.70000,,-inf,inf,Å
3,lam7o3,cell,,length_c,5.50000,,-inf,inf,Å
4,lam7o3,atom_site,La,fract_x,0.48000,,-inf,inf,
5,lam7o3,atom_site,La,fract_z,0.00400,,-inf,inf,
6,lam7o3,atom_site,La,adp_iso,0.10000,,-inf,inf,Å²
7,lam7o3,atom_site,Ti,adp_iso,0.10000,,-inf,inf,Å²
8,lam7o3,atom_site,O1,fract_x,0.51000,,-inf,inf,
9,lam7o3,atom_site,O1,fract_z,0.57000,,-inf,inf,
10,lam7o3,atom_site,O1,adp_iso,0.10000,,-inf,inf,Å²


### Set Constraints

Create aliases for the constrained Biso parameters. The seven
elements share one B site and therefore one Biso value; O1 and O2
are also modeled with one shared Biso value.

In [25]:
# B sites: Ti, Cr, Mn, Fe, Co, Ni, Cu
project.analysis.aliases.create(
    id='biso_Ti',
    param=structure.atom_sites['Ti'].adp_iso,
)
project.analysis.aliases.create(
    id='biso_Cr',
    param=structure.atom_sites['Cr'].adp_iso,
)
project.analysis.aliases.create(
    id='biso_Mn',
    param=structure.atom_sites['Mn'].adp_iso,
)
project.analysis.aliases.create(
    id='biso_Fe',
    param=structure.atom_sites['Fe'].adp_iso,
)
project.analysis.aliases.create(
    id='biso_Co',
    param=structure.atom_sites['Co'].adp_iso,
)
project.analysis.aliases.create(
    id='biso_Ni',
    param=structure.atom_sites['Ni'].adp_iso,
)
project.analysis.aliases.create(
    id='biso_Cu',
    param=structure.atom_sites['Cu'].adp_iso,
)

# O sites: O1, O2
project.analysis.aliases.create(
    id='biso_O1',
    param=structure.atom_sites['O1'].adp_iso,
)
project.analysis.aliases.create(
    id='biso_O2',
    param=structure.atom_sites['O2'].adp_iso,
)

Apply the equality constraints using the aliases.

In [26]:
project.analysis.constraints.create(id='1', expression='biso_Cr = biso_Ti')
project.analysis.constraints.create(id='2', expression='biso_Mn = biso_Ti')
project.analysis.constraints.create(id='3', expression='biso_Fe = biso_Ti')
project.analysis.constraints.create(id='4', expression='biso_Co = biso_Ti')
project.analysis.constraints.create(id='5', expression='biso_Ni = biso_Ti')
project.analysis.constraints.create(id='6', expression='biso_Cu = biso_Ti')

project.analysis.constraints.create(id='7', expression='biso_O2 = biso_O1')

Display the defined constraints.

In [27]:
project.analysis.display.constraints()

User defined constraints


,id,expression
1,1,biso_Cr = biso_Ti
2,2,biso_Mn = biso_Ti
3,3,biso_Fe = biso_Ti
4,4,biso_Co = biso_Ti
5,5,biso_Ni = biso_Ti
6,6,biso_Cu = biso_Ti
7,7,biso_O2 = biso_O1


Constraints enabled: True


### Fit Initial Model

The first fit uses a background estimated directly from the measured
pattern.

#### Display Pattern (Before Fit)

Compare the measured pattern with the calculation from the starting
model before optimization.

In [28]:
project.display.pattern(expt_name='p021')

In [29]:
project.display.pattern(expt_name='p021', x_min=2.2, x_max=4.0)

#### Run Fitting

In [30]:
project.analysis.minimizer.chi_square_change_tolerance = 1e-2

In [31]:
project.analysis.fit()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'p021' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


,iteration,time (s),χ²,change / status
1,1,0.25,276.18,
2,17,5.46,276.19,
3,34,10.65,276.18,
4,39,12.02,46.46,83.2% ↓
5,56,17.03,46.46,
6,73,22.34,46.46,
7,75,22.77,15.97,65.6% ↓
8,93,28.24,15.97,
9,110,33.27,15.97,
10,111,33.48,3.89,75.7% ↓


🏆 Best goodness-of-fit (reduced χ²) is 0.88 at iteration 257


✅ Fitting complete.


Display the initial fit summary and the strongest parameter
correlations.

In [32]:
project.display.fit.results()

⚙️ Settings used:


,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.
2,chi_square_change_tolerance,0.01,Relative change in the objective (chi-square) used to stop fitting.
3,parameter_change_tolerance,1e-08,Relative change in fitted parameters used to stop fitting.
4,gradient_tolerance,0.0,Gradient orthogonality used to stop fitting; zero disables it.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),91.03
4,🔁 Iterations,255
5,📏 Goodness-of-fit (reduced χ²),0.88
6,"📏 R-factor (Rf, %)",2.39
7,"📏 R-factor squared (Rf², %)",2.56
8,"📏 Weighted R-factor (wR, %)",3.58


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,lam7o3,cell,,length_a,Å,5.5000,5.4623,0.0002,0.69 % ↓
2,lam7o3,cell,,length_b,Å,7.7000,7.7332,0.0004,0.43 % ↑
3,lam7o3,cell,,length_c,Å,5.5000,5.4903,0.0002,0.18 % ↓
4,lam7o3,atom_site,La,fract_x,,0.4800,0.4766,0.0002,0.72 % ↓
5,lam7o3,atom_site,La,fract_z,,0.0040,0.0053,0.0006,31.90 % ↑
6,lam7o3,atom_site,La,adp_iso,Å²,0.1000,0.5939,0.0217,493.90 % ↑
7,lam7o3,atom_site,Ti,adp_iso,Å²,0.1000,0.3424,0.0291,242.42 % ↑
8,lam7o3,atom_site,O1,fract_x,,0.5100,0.5070,0.0021,0.59 % ↓
9,lam7o3,atom_site,O1,fract_z,,0.5700,0.5582,0.0035,2.07 % ↓
10,lam7o3,atom_site,O1,adp_iso,Å²,0.1000,0.5917,0.1594,491.68 % ↑


In [33]:
project.display.fit.correlations(max_parameters=5)

#### Display Pattern (After Initial Fit)

Inspect the full fitted pattern and the nonuniform low-angle
background region.

In [34]:
project.display.pattern(expt_name='p021')

In [35]:
project.display.pattern(expt_name='p021', x_min=2.2, x_max=4.0)

### Improve Background Estimate

With a fitted peak model available, repeat automatic estimation using
the calculated peak contribution. This replaces the original points
with a model-guided estimate. Replacement points are fixed by default,
so mark their intensities free again.

In [36]:
experiment.background.auto_estimate(use_model=True)

In [37]:
experiment.background.show()

Line-segment background points


,Position,Intensity
1,2.00330,576.07579
2,2.17160,638.44269
3,2.23470,662.65186
4,2.29780,695.09265
5,2.40300,752.13934
6,2.75000,948.73561
7,2.79210,960.48590
8,2.83410,967.24257
9,2.87620,968.71012
10,2.91820,964.89609


In [38]:
for point in experiment.background:
    point.intensity.free = True

#### Run Fitting

In [39]:
project.analysis.fit()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'p021' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


,iteration,time (s),χ²,change / status
1,1,0.60,2.77,
2,17,5.79,2.77,
3,34,11.30,2.77,
4,50,16.58,2.77,
5,66,21.80,2.77,
6,69,22.44,0.16,94.2% ↓
7,86,27.51,0.16,
8,103,32.91,0.16,
9,120,37.94,0.16,
10,136,50.52,0.16,


🏆 Best goodness-of-fit (reduced χ²) is 0.16 at iteration 135


✅ Fitting complete.


In [40]:
project.display.fit.results()

⚙️ Settings used:


,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.
2,chi_square_change_tolerance,0.01,Relative change in the objective (chi-square) used to stop fitting.
3,parameter_change_tolerance,1e-08,Relative change in fitted parameters used to stop fitting.
4,gradient_tolerance,0.0,Gradient orthogonality used to stop fitting; zero disables it.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),50.52
4,🔁 Iterations,133
5,📏 Goodness-of-fit (reduced χ²),0.16
6,"📏 R-factor (Rf, %)",1.01
7,"📏 R-factor squared (Rf², %)",1.61
8,"📏 Weighted R-factor (wR, %)",1.51


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,lam7o3,cell,,length_a,Å,5.4623,5.4622,0.0001,0.00 % ↓
2,lam7o3,cell,,length_b,Å,7.7332,7.7331,0.0002,0.00 % ↓
3,lam7o3,cell,,length_c,Å,5.4903,5.4902,0.0001,0.00 % ↓
4,lam7o3,atom_site,La,fract_x,,0.4766,0.4767,0.0001,0.03 % ↑
5,lam7o3,atom_site,La,fract_z,,0.0053,0.0045,0.0003,15.62 % ↓
6,lam7o3,atom_site,La,adp_iso,Å²,0.5939,0.6260,0.0095,5.40 % ↑
7,lam7o3,atom_site,Ti,adp_iso,Å²,0.3424,0.3377,0.0126,1.38 % ↓
8,lam7o3,atom_site,O1,fract_x,,0.5070,0.5083,0.0010,0.25 % ↑
9,lam7o3,atom_site,O1,fract_z,,0.5582,0.5707,0.0016,2.24 % ↑
10,lam7o3,atom_site,O1,adp_iso,Å²,0.5917,0.3587,0.0660,39.37 % ↓


In [41]:
project.display.fit.correlations(max_parameters=5)

#### Display Pattern (After Final Fit)

In [42]:
project.display.pattern(expt_name='p021')

In [43]:
project.display.pattern(expt_name='p021', x_min=2.2, x_max=4.0)

## 📊 Report

The HTML report is written automatically when the project is saved.
PDF generation can be enabled before the final save when required.

In [44]:
# Enable PDF report generation before the last save (time consuming)
# project.report.pdf = True

## 💾 Save Project

In [45]:
project.save()

Saving project 📦 'lam7o3_p021' to '../../../projects/refine-lam7o3-p021'


├── 📄 project.edi
├── 📁 structures/
│   └── 📄 lam7o3.edi
├── 📁 experiments/
│   └── 📄 p021.edi
├── 📁 analysis/
│   └── 📄 analysis.edi
└── 📁 reports/
    └── 📄 lam7o3_p021.html
